This notebook reads the title and abstract decisions and the full-text decisions exported from the screening tool, checks them against the deduplicated records, and counts the sources at each stage. It adds the `screening`, `included`, and `by_block` sections to `data/processed/screening_counts.yaml`. [04 Synthesis](04-synthesis.ipynb) draws the flow diagram from that file.

The export formats are in the [screening guide](../materials/screening-guide.qmd). A record is a title and abstract. A report is a full-text document. A source of evidence can have several reports.

In [1]:
from pathlib import Path

import pandas as pd
import yaml

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").exists())
PROC = ROOT / "data" / "processed"
EXP = ROOT / "data" / "raw" / "screening_exports"
dedup_path = PROC / "deduplicated_records.csv"
ta_path = EXP / "title_abstract.csv"
ft_path = EXP / "full_text.csv"
counts_path = PROC / "screening_counts.yaml"

missing = [str(p.relative_to(ROOT)) for p in (dedup_path, ta_path) if not p.exists()]
have_data = not missing
if not have_data:
    print("Not run yet. Missing: " + ", ".join(missing))


def read(path, required, optional=()):
    df = pd.read_csv(path, dtype=str).fillna("")
    df.columns = [c.strip().lower() for c in df.columns]
    absent = [c for c in required if c not in df.columns]
    if absent:
        raise ValueError(f"{path.name} is missing column(s): {', '.join(absent)}")
    for c in optional:
        if c not in df.columns:
            df[c] = ""
    return df.apply(lambda s: s.str.strip())

Not run yet. Missing: data/processed/deduplicated_records.csv, data/raw/screening_exports/title_abstract.csv


In [2]:
if have_data:
    dedup = read(dedup_path, ["record_id"], ["blocks"])
    ta = read(ta_path, ["record_id", "decision"])
    ta["decision"] = ta["decision"].str.lower()
    ft_cols = ["record_id", "source_id", "retrieved", "decision", "reason"]
    if ft_path.exists():
        ft = read(ft_path, ["record_id", "retrieved", "decision", "reason"], ["source_id"])
    else:
        ft = pd.DataFrame({c: pd.Series(dtype=str) for c in ft_cols})
    for col in ["retrieved", "decision", "reason"]:
        ft[col] = ft[col].str.lower()
    ft["source_id"] = ft["source_id"].where(ft["source_id"] != "", ft["record_id"])

    ta_ok = ta[ta["decision"].isin(["include", "exclude"])]
    screened = len(ta_ok)
    ta_excluded = int((ta_ok["decision"] == "exclude").sum())
    sought = int((ta_ok["decision"] == "include").sum())

    not_retrieved = int((ft["retrieved"] == "no").sum())
    ft_ok = ft[(ft["retrieved"] == "yes") & ft["decision"].isin(["include", "exclude"])]
    assessed = len(ft_ok)
    ft_excl = ft_ok[ft_ok["decision"] == "exclude"]
    ft_incl = ft_ok[ft_ok["decision"] == "include"]
    reasons = ft_excl["reason"].replace("", "unspecified").value_counts()
    included_reports = len(ft_incl)
    included_sources = int(ft_incl["source_id"].nunique())

## Checks

In [3]:
if have_data:
    problems = []
    unknown = set(ta["record_id"]) - set(dedup["record_id"])
    if unknown:
        problems.append(f"{len(unknown)} title/abstract record_id value(s) are not in deduplicated_records.csv.")
    unscreened = len(set(dedup["record_id"]) - set(ta["record_id"]))
    if unscreened:
        problems.append(f"{unscreened} deduplicated record(s) have no title/abstract decision yet.")
    if ta["record_id"].duplicated().any():
        problems.append("Some record_id values appear more than once in title_abstract.csv.")
    bad = ta[~ta["decision"].isin(["include", "exclude"])]
    if len(bad):
        problems.append(f"{len(bad)} title/abstract row(s) have a decision other than include or exclude.")
    stray = set(ft["record_id"]) - set(ta_ok.loc[ta_ok["decision"] == "include", "record_id"])
    if stray:
        problems.append(f"{len(stray)} full-text record_id value(s) were not included at title and abstract.")
    pending = sought - not_retrieved - assessed
    if pending > 0:
        problems.append(f"{pending} included record(s) have no full-text outcome yet.")
    if (ft_excl["reason"] == "").any():
        problems.append("Some excluded full-text rows have no reason.")
    print("\n".join(problems) if problems else "No problems found.")

## Counts

In [4]:
if have_data:
    summary = pd.DataFrame(
        [
            ("Records screened", screened),
            ("Records excluded at title and abstract", ta_excluded),
            ("Reports sought for retrieval", sought),
            ("Reports not retrieved", not_retrieved),
            ("Reports assessed for eligibility", assessed),
            ("Reports excluded at full text", len(ft_excl)),
            ("Reports included", included_reports),
            ("Sources of evidence included", included_sources),
        ],
        columns=["Stage", "n"],
    )
    display(summary)
    if len(reasons):
        display(reasons.rename_axis("Exclusion reason").reset_index(name="n"))

In [5]:
if have_data:
    block_ids = {}
    for rid, blocks in zip(dedup["record_id"], dedup["blocks"]):
        for b in [x.strip() for x in blocks.split(";") if x.strip()]:
            block_ids.setdefault(b, set()).add(rid)
    ta_excl_ids = ta_ok.loc[ta_ok["decision"] == "exclude", "record_id"]
    by_block = {}
    for b, ids in sorted(block_ids.items()):
        by_block[b] = {
            "records_identified": len(ids),
            "records_screened": int(ta_ok["record_id"].isin(ids).sum()),
            "records_excluded": int(ta_excl_ids.isin(ids).sum()),
            "reports_assessed": int(ft_ok["record_id"].isin(ids).sum()),
            "reports_excluded": int(ft_excl["record_id"].isin(ids).sum()),
            "reports_included": int(ft_incl["record_id"].isin(ids).sum()),
        }

    counts = (yaml.safe_load(counts_path.read_text()) or {}) if counts_path.exists() else {}
    counts["screening"] = {
        "records_screened": screened,
        "records_excluded": ta_excluded,
        "reports_sought": sought,
        "reports_not_retrieved": not_retrieved,
        "reports_assessed": assessed,
        "reports_excluded": [{"reason": r, "n": int(n)} for r, n in reasons.items()],
    }
    counts["included"] = {"sources": included_sources, "reports": included_reports}
    counts["by_block"] = by_block
    counts_path.write_text(yaml.safe_dump(counts, sort_keys=False))
    print("Wrote data/processed/screening_counts.yaml")
    if by_block:
        display(pd.DataFrame(by_block).T.rename_axis("block"))